## Feature Engineering

Creating new predictors from the baseline ones, can be benifical to imrpove the correlation between features and target value (day ahead price)

The top 5 new engineered predictors that would most likely help the ML explain variations in electricity prices are: 

1. Historical price average 
2. Historical price volatility 
3. Hydropower generation share 
4. Residual electricity demand 
5. Heating degree indicator 

NOTE: Also comment on training on ML models- all predictors we have that are known at the same time as target value should not be used to train the model as this will cause data leakage. We will be using the forecasted values and the lagged values only to train the model to get realistic results that will work more optimal on real data, and that can serve real purpose on valuation problems regarding electricity price forecasting. 

In [17]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import importlib
importlib.reload(feature_eng)

import os



from Alexander_work_folder.Feature_eng import feature_eng

df = feature_eng.load_data()

print(df.head())

                  timestamp  day_ahead_price  load_forecast  actual_load  \
0 2019-01-31 23:00:00+00:00            51.31        5785.04      5834.83   
1 2019-02-01 00:00:00+00:00            50.47        5641.08      5705.04   
2 2019-02-01 01:00:00+00:00            50.03        5459.89      5508.91   
3 2019-02-01 02:00:00+00:00            50.10        5473.32      5538.59   
4 2019-02-01 03:00:00+00:00            50.72        5524.60      5589.15   

   wind_forecast  hydro_run_of_river  hydro_reservoir  hour  day_of_week  \
0          66.83              585.48          1296.87     0            4   
1          78.01              580.90          1291.26     1            4   
2          75.99              579.13          1293.26     2            4   
3          73.48              577.29          1291.11     3            4   
4          72.09              577.19          1289.16     4            4   

   month  ...  price_lag_168h  actual_load_lag_24h  actual_load_lag_168h  \
0      2  

### 1. Historical price average

Rather than relying on a single previous price, we can combine prices from the same hour over several previous days.

This should help reduce sensitivity to unusual price movements on individual days and provide a more stable estimate of the prevailing price level.

$P_{avg} = \frac{P_{t-24}+P_{t-48}+P_{t-168}}{3}$

It should help nonlinear models identify broader pricing patterns.

### 2. Historical price volatility

Calculate the standard deviation of the same three historical prices.

$P_{std} = std(P_{t-24}, P_{t-48}, P_{t-168})$

Electricity prices can change dramatically during periods of market instability, like 2025, this can make our model more sturdy towards years with high unpredictability.

The feature captures differences between recent prices, which may indicate changing market conditions. It could be especially useful for identifying periods with unusually high prices, such as those observed during the European energy crisis (2025). 

### 3. Hydropower generation share (historical) 

Calculate the proportion of forecast electricity demand represented by hydropower generation. This would be the hydropower generation share generated a week before. 

$H_{share}(t-168) = \frac{H_{RoR,t-168}+H_{Reservoir,t-168}}{L_{actual,t-168}}$

Hydropower is particularly relevant for Norwegian electricity markets. Higher hydropower generation relative to demand can indicate more abundant electricity supply and potentially lower prices.

This feature combines generation and demand into one variable, allowing the model to learn their relationship rather than treating them as entirely independent predictors.

### 4. Residual electricity demand 

Determine the electricity demand remaining after subtracting the generation variables available in the dataset. 
This is a simplified measure of the demand that must be covered by other generation sources or imports.

When residual demand is high, the market may require more expensive marginal electricity supply, potentially increasing the day-ahead price.

$L_{residual,t} = L_{forecast,t} - W_{forecast,t} - H_{RoR,t-168} - H_{reservoir,t-168}$

This is economically meaningful, although it does not account for transmission constraints, imports, exports, or generation outside the included variables.

The lagged hydropower variables provide an approximation of expected hydropower generation based on historical patterns.

However, this is only a proxy for residual demand, not the actual expected residual demand, because hydropower generation can change substantially between weeks

### 5. Heating degree indicator 

Electricity demand in Norway is strongly influenced by temperature, , particularly during cold periods.

Rather than using temperature directly, create a nonlinear heating indicator:

$HDI(t) = max(0, 15-T_{NO1})$

Here, 15°C is an initial threshold that should be tuned using training data. Here, when temerpature is higher than 15 degrees the HDI is set to 0. A higher HDI would indicate lower temperatures than 15 degrees, giving lower temperatures more significant correlation with target value. 

The feature distinguishes between temperatures that are likely to generate heating demand and warmer temperatures where the heating effect is smaller.

In [18]:
df = feature_eng.add_features(df)
print(df.head())

                  timestamp  day_ahead_price  load_forecast  actual_load  \
0 2019-01-31 23:00:00+00:00            51.31        5785.04      5834.83   
1 2019-02-01 00:00:00+00:00            50.47        5641.08      5705.04   
2 2019-02-01 01:00:00+00:00            50.03        5459.89      5508.91   
3 2019-02-01 02:00:00+00:00            50.10        5473.32      5538.59   
4 2019-02-01 03:00:00+00:00            50.72        5524.60      5589.15   

   wind_forecast  hydro_run_of_river  hydro_reservoir  hour  day_of_week  \
0          66.83              585.48          1296.87     0            4   
1          78.01              580.90          1291.26     1            4   
2          75.99              579.13          1293.26     2            4   
3          73.48              577.29          1291.11     3            4   
4          72.09              577.19          1289.16     4            4   

   month  ...  hydro_reservoir_lag_24h  hydro_reservoir_lag_168h  \
0      2  ...     

In [19]:
# saving new feautred engineered dataset to the data folder: 

path = f"C:/Users/alexa/Documents/GitHub/ML-project-electricity-forecasting/Alexander_work_folder/Feature_eng/Feature_eng_dataset/NO1_ML_dataset_2019_2025_feature_eng.csv"

feature_eng.save_feature_engineered_data(df, path)

